# Chapitre 5 — Nettoyage des données

**Durée estimée : 10-12 heures**

---

## Objectifs d'apprentissage

À la fin de ce chapitre, vous serez capable de :

1. **Appliquer** différentes stratégies de traitement des valeurs manquantes (suppression, imputation)
2. **Identifier et supprimer** les doublons en préservant les informations pertinentes
3. **Traiter** les valeurs aberrantes selon le contexte métier
4. **Nettoyer** les types de données (dates, numériques, texte) pour les rendre exploitables

---

## 5.5 Nettoyage de texte

### Opérations de base

In [1]:
import pandas as pd

# Données textuelles désordonnées
df_texte = pd.DataFrame({
    'nom': ['  Jean DUPONT  ', 'marie-claire Martin', 'PIERRE durand', 'émilie Côté'],
    'email': ['Jean.Dupont@Gmail.COM', 'marie@test.fr', 'PIERRE123@yahoo.fr', 'emilie@test'],
    'telephone': ['06 12 34 56 78', '0687654321', '+33 6 11 22 33 44', '06-99-88-77-66'],
    'adresse': ['12 rue de Paris, 75001 PARIS', '5 avenue Lyon 69001', 'Marseille', '10 bd Bordeaux 33000']
})

print("Avant nettoyage :")
print(df_texte)

Avant nettoyage :
                   nom                  email          telephone  \
0        Jean DUPONT    Jean.Dupont@Gmail.COM     06 12 34 56 78   
1  marie-claire Martin          marie@test.fr         0687654321   
2        PIERRE durand     PIERRE123@yahoo.fr  +33 6 11 22 33 44   
3          émilie Côté            emilie@test     06-99-88-77-66   

                        adresse  
0  12 rue de Paris, 75001 PARIS  
1           5 avenue Lyon 69001  
2                     Marseille  
3          10 bd Bordeaux 33000  


In [ ]:
# Supprimer les espaces et normaliser la casse
df_texte['nom_clean'] = df_texte['nom'].str.strip().str.title()
# str.strip() enlève les espaces avant et après
# str.title() met en majuscule la première lettre de chaque mot et le reste en minuscule

print("Noms nettoyés (strip + title) :")
print(df_texte['nom_clean'])

Noms nettoyés (strip + title) :
0            Jean Dupont
1    Marie-Claire Martin
2          Pierre Durand
3            Émilie Côté
Name: nom_clean, dtype: object


In [3]:
# Normaliser les emails (lower, strip)
df_texte['email_clean'] = df_texte['email'].str.lower().str.strip()
# str.lower() met tout en minuscule

print("\nEmails nettoyés :")
print(df_texte['email_clean'])


Emails nettoyés :
0    jean.dupont@gmail.com
1            marie@test.fr
2       pierre123@yahoo.fr
3              emilie@test
Name: email_clean, dtype: object


In [ ]:
# Nettoyer les téléphones (garder uniquement les chiffres)
df_texte['tel_clean'] = df_texte['telephone'].str.replace(r'[^\d]', '', regex=True)
# r '[^\d]' correspond à tout ce qui n'est pas un chiffre 
# d = digit (chiffre)

# Normaliser au format français (commencer par 0)
df_texte['tel_clean'] = df_texte['tel_clean'].str.replace('^33', '0', regex=True)

print("\nTéléphones nettoyés :")
print(df_texte['tel_clean'])

### Expressions régulières

In [ ]:
import re

# Extraire le code postal de l'adresse
df_texte['code_postal'] = df_texte['adresse'].str.extract(r'(\d{5})')
print("Codes postaux extraits :")
print(df_texte[['adresse', 'code_postal']])

In [ ]:
# Valider le format email
pattern_email = r'^[\w\.-]+@[\w\.-]+\.\w+$'
df_texte['email_valide'] = df_texte['email_clean'].str.match(pattern_email)
print("\nValidation des emails :")
print(df_texte[['email_clean', 'email_valide']])

### Standardisation des catégories

In [ ]:
# Problème courant : variations d'écriture
df_pays = pd.DataFrame({
    'pays': ['France', 'france', 'FRANCE', 'FR', 'francia', 'Allemagne', 'DE', 'germany']
})

print("Variations de pays :")
print(df_pays['pays'].unique())

In [ ]:
# Solution : mapping
mapping_pays = {
    'france': 'France',
    'fr': 'France',
    'francia': 'France',
    'allemagne': 'Allemagne',
    'de': 'Allemagne',
    'germany': 'Allemagne'
}

df_pays['pays_clean'] = df_pays['pays'].str.lower().map(mapping_pays).fillna(df_pays['pays'])
print("\nPays standardisés :")
print(df_pays)


### Les Commandes Pandas `.str` (Les Indispensables)

Voici les méthodes que tu utiliseras dans 90% des cas.

| Commande | Action | Exemple d'usage |
| --- | --- | --- |
| **`.str.lower()` / `.str.upper()`** | Met tout en minuscules / majuscules. | Standardiser avant de comparer `("Paris" == "paris")`. |
| **`.str.strip()`** | Enlève les espaces inutiles au début et à la fin. | Nettoyer `" Jean "` en `"Jean"`. |
| **`.str.replace('A', 'B')`** | Remplace un caractère ou un motif par un autre. | Enlever les symboles : `.str.replace('€', '')`. |
| **`.str.contains('motif')`** | Cherche si un motif existe (Renvoie `True`/`False`). | Filtrer les emails gmail : `.str.contains('@gmail')`. |
| **`.str.split(' ')`** | Coupe le texte selon un séparateur. | Séparer Prénom/Nom : `.str.split(' ')`. |
| **`.str.len()`** | Calcule la longueur du texte. | Détecter les codes postaux invalides (`len != 5`). |
| **`.str.extract(r'(regex)')`** | Extrait une partie précise du texte via Regex. | Extraire juste le domaine d'un email. |

---

### Le "Cheat Sheet" Regex (Expressions Régulières)

Quand le nettoyage simple ne suffit pas, on utilise les **Regex** à l'intérieur de `.str.replace()`, `.str.contains()` ou `.str.extract()`.

#### Les Caractères Spéciaux (Les "Jokers")

* `^` : Début de la chaîne (ex: `^06` -> commence par 06).
* `$` : Fin de la chaîne (ex: `fr$` -> finit par fr).
* `.` : N'importe quel caractère (sauf retour à la ligne).
* `|` : OU logique (ex: `Paris|Lyon` -> Paris OU Lyon).
* `\` : Échappement (pour chercher un vrai point, on écrit `\.`).

#### Les Classes de Caractères (Ce qu'on cherche)

* `\d` : Un chiffre (0 à 9).
* `\D` : Tout sauf un chiffre.
* `\w` : Une lettre ou un chiffre (alphanumérique).
* `\s` : Un espace (espace, tabulation, retour ligne).
* `[a-z]` : Une lettre minuscule.
* `[A-Z]` : Une lettre majuscule.

#### Les Quantificateurs (Combien de fois ?)

* `+` : 1 fois ou plus (ex: `\d+` -> un nombre entier comme "12" ou "4500").
* `*` : 0 fois ou plus.
* `?` : 0 ou 1 fois (optionnel).
* `{n}` : Exactement n fois (ex: `\d{5}` -> code postal de 5 chiffres).

### ✍️ Exercice 5.6 : Nettoyage de texte complet (20 min)

In [ ]:
import pandas as pd

# Données textuelles désordonnées
df_ex6 = pd.DataFrame({
    'nom': ['  Jean DUPONT  ', 'marie-claire Martin', 'PIERRE durand', 'émilie Côté'],
    'email': ['Jean.Dupont@Gmail.COM', 'marie@test.fr', 'PIERRE123@yahoo.fr', 'emilie@test'],
    'telephone': ['06 12 34 56 78', '0687654321', '+33 6 11 22 33 44', '06-99-88-77-66'],
    'adresse': ['12 rue de Paris, 75001 PARIS', '5 avenue Lyon 69001', 'Marseille', '10 bd Bordeaux 33000']
})

print("Avant nettoyage :")
print(df_ex6)

In [ ]:
# 1. Normaliser les noms (strip, title case)
df_ex6['nom_clean'] = df_ex6['nom'].str.strip().str.title()

# 2. Normaliser les emails (lower, strip)
df_ex6['email_clean'] = df_ex6['email'].str.lower().str.strip()

# 3. Valider les emails (contient @ et .)
df_ex6['email_valide'] = df_ex6['email_clean'].str.contains('@') & df_ex6['email_clean'].str.contains(r'\.')

# 4. Nettoyer les téléphones (garder uniquement les chiffres)
df_ex6['tel_clean'] = df_ex6['telephone'].str.replace(r'[^\d]', '', regex=True)
# Normaliser au format français (commencer par 0)
df_ex6['tel_clean'] = df_ex6['tel_clean'].str.replace('^33', '0', regex=True)

# 5. Extraire le code postal de l'adresse
df_ex6['code_postal'] = df_ex6['adresse'].str.extract(r'(\d{5})')

print("\nAprès nettoyage :")
print(df_ex6[['nom_clean', 'email_clean', 'email_valide', 'tel_clean', 'code_postal']])